In [0]:
%pip install databricks-labs-dqx

In [0]:
%restart_python

In [0]:
%run ./00_setup

In [0]:
# Purpose: verify the installed DQX version and initialize
# the DQX engine using Databricks notebook authentication.

from importlib.metadata import version

from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

dqx_version = version(
    "databricks-labs-dqx"
)

workspace_client = WorkspaceClient()
dq_engine = DQEngine(workspace_client)

print("DQX version:", dqx_version)
print("DQX engine initialized successfully")
print("Catalog:", catalog)
print("Schema:", schema)

In [0]:
# Purpose: load the Gold order-item fact table that DQX will monitor.

dqx_input_table = (
    f"{catalog}.{schema}.gold_fact_order_items"
)

if not spark.catalog.tableExists(dqx_input_table):
    raise RuntimeError(
        f"DQX input table does not exist: {dqx_input_table}"
    )

dqx_input_df = spark.table(
    dqx_input_table
)

dqx_input_rows = dqx_input_df.count()

print("DQX input table:", dqx_input_table)
print("DQX input rows:", dqx_input_rows)

display(
    dqx_input_df.limit(10)
)

In [0]:
# Purpose: define explicit DQX rules for required fields,
# monetary ranges, and surrogate-key uniqueness.

from databricks.labs.dqx.engine import DQEngine

required_string_columns = [
    "order_item_sk",
    "order_id",
    "customer_sk",
    "product_sk",
    "seller_sk",
    "customer_id",
    "product_id",
    "seller_id",
    "order_status",
]

# Create one explicit rule for every required string column.
dqx_checks = [
    {
        "name": f"{column}_required",
        "criticality": "error",
        "check": {
            "function": "is_not_null_and_not_empty",
            "arguments": {
                "column": column,
                "trim_strings": True,
            },
        },
    }
    for column in required_string_columns
]

# Add rules that are not string-based.
dqx_checks.extend(
    [
        {
            "name": "date_sk_required",
            "criticality": "error",
            "check": {
                "function": "is_not_null",
                "arguments": {
                    "column": "date_sk",
                },
            },
        },
        {
            "name": "valid_price_range",
            "criticality": "error",
            "check": {
                "function": "is_in_range",
                "arguments": {
                    "column": "price",
                    "min_limit": 0,
                    "max_limit": 100000,
                },
            },
        },
        {
            "name": "valid_freight_range",
            "criticality": "error",
            "check": {
                "function": "is_in_range",
                "arguments": {
                    "column": "freight_value",
                    "min_limit": 0,
                    "max_limit": 100000,
                },
            },
        },
        {
            "name": "valid_item_total_range",
            "criticality": "error",
            "check": {
                "function": "is_in_range",
                "arguments": {
                    "column": "item_total_value",
                    "min_limit": 0,
                    "max_limit": 200000,
                },
            },
        },
        {
            "name": "unique_order_item_surrogate_key",
            "criticality": "error",
            "check": {
                "function": "is_unique",
                "arguments": {
                    "columns": [
                        "order_item_sk",
                    ],
                },
            },
        },
    ]
)

dqx_rule_validation = DQEngine.validate_checks(
    dqx_checks
)

print("Defined DQX rules:", len(dqx_checks))
print("DQX rule validation:", dqx_rule_validation)

if dqx_rule_validation.has_errors:
    raise RuntimeError(
        "Invalid DQX rule configuration: "
        + str(dqx_rule_validation.errors)
    )

print("PASS: all DQX rules are valid")

In [0]:
# Purpose: apply the validated DQX rules and split the Gold fact
# into valid records and records requiring quarantine.

dqx_valid_df, dqx_quarantine_df = (
    dq_engine.apply_checks_by_metadata_and_split(
        dqx_input_df,
        dqx_checks,
    )
)

# Cache because both DataFrames are evaluated more than once.
dqx_valid_df = dqx_valid_df.cache()
dqx_quarantine_df = dqx_quarantine_df.cache()

dqx_valid_rows = dqx_valid_df.count()
dqx_quarantine_rows = dqx_quarantine_df.count()

dqx_reconciled_rows = (
    dqx_valid_rows
    + dqx_quarantine_rows
)

dqx_reconciliation_pass = (
    dqx_reconciled_rows
    == dqx_input_rows
)

print("DQX input rows:", dqx_input_rows)
print("DQX valid rows:", dqx_valid_rows)
print("DQX quarantine rows:", dqx_quarantine_rows)
print("DQX reconciled rows:", dqx_reconciled_rows)

print(
    "DQX reconciliation:",
    "PASS" if dqx_reconciliation_pass else "FAIL",
)

print("Quarantined records and DQX error details:")

display(
    dqx_quarantine_df.limit(20)
)

In [0]:
# Purpose: save the current DQX quarantine result.
# This table is overwritten because it represents the latest snapshot.

from pyspark.sql import functions as F

dqx_quarantine_table = (
    f"{catalog}.{schema}."
    "dqx_gold_fact_order_items_quarantine"
)

dqx_quarantine_output_df = (
    dqx_quarantine_df
    .withColumn(
        "_dqx_checked_by",
        F.expr("session_user()"),
    )
    .withColumn(
        "_dqx_checked_at",
        F.current_timestamp(),
    )
)

(
    dqx_quarantine_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dqx_quarantine_table)
)

print(
    "DQX quarantine table:",
    dqx_quarantine_table,
)

print(
    "Saved quarantine rows:",
    spark.table(dqx_quarantine_table).count(),
)

In [0]:
# Purpose: save a historical DQX monitoring summary for every run.

dqx_quality_pass = (
    dqx_reconciliation_pass
    and dqx_quarantine_rows == 0
)

dqx_quality_status = (
    "PASS"
    if dqx_quality_pass
    else "FAIL"
)

dqx_audit_df = spark.createDataFrame(
    [
        (
            dqx_input_table,
            dqx_input_rows,
            dqx_valid_rows,
            dqx_quarantine_rows,
            dqx_reconciled_rows,
            len(dqx_checks),
            dqx_quality_status,
        )
    ],
    """
        input_table STRING,
        input_rows LONG,
        valid_rows LONG,
        quarantine_rows LONG,
        reconciled_rows LONG,
        configured_rules INT,
        status STRING
    """,
)

dqx_audit_df = (
    dqx_audit_df
    .withColumn(
        "checked_by",
        F.expr("session_user()"),
    )
    .withColumn(
        "_checked_at",
        F.current_timestamp(),
    )
)

dqx_audit_table = (
    f"{catalog}.{schema}."
    "dqx_gold_fact_order_items_audit"
)

(
    dqx_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(dqx_audit_table)
)

display(
    spark.table(dqx_audit_table)
    .orderBy(F.col("_checked_at").desc())
    .limit(10)
)

if not dqx_quality_pass:
    raise RuntimeError(
        f"DQX validation failed with "
        f"{dqx_quarantine_rows} quarantined rows."
    )

print("SUCCESS: DQX quality monitoring passed")